
# EduCodeInsight OOP Pipeline

This notebook is a  pipeline for **EduCodeInsight: Enhancing student feedback through automated error analysis**.

## OOP structure

- `FeedbackParser`: parses student feedback into domain objects.
- `FeedbackValidator` : alidates uploaded feedback structure.
- `AssignmentContext` :xtracts and matches memo/specification context.
- `LLMClassifier` : encapsulates Together AI classification and usage tracking.
- `AnalysisEngine` : creates DataFrames, category summaries, recurring-error analysis and overall analysis.
- `ChartGenerator`  : creates report/dashboard figures.
- `ReportGenerator` : creates concise visual DOCX reports.
- `ResultManager` : saves/loads JSON results.
- `EduCodeInsightPipeline` : orchestrates the services for an assignment.


In [ ]:
%pip install -q pandas matplotlib pdfplumber requests pydantic python-docx

## 1. Import the OOP pipeline

In [ ]:
# Import the OOP classes from separate Python modules.
# The package mirrors a C#-style class-per-file structure.

from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pipeline").exists() and (PROJECT_ROOT.parent / "pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline import (
    CATEGORIES,
    MODEL_CONFIGS,
    DEFAULT_MODEL_LABEL,
    DEFAULT_MODEL,
    MAX_WORKERS,
    BATCH_SIZE,
    MAX_RETRIES,
    REQUEST_TIMEOUT,
    RubricItem,
    QuestionBlock,
    StudentFeedback,
    MemoSection,
    ClassificationResult,
    FeedbackParser,
    FeedbackValidator,
    AssignmentContext,
    LLMClassifier,
    AnalysisEngine,
    ChartGenerator,
    ReportGenerator,
    ResultManager,
    EduCodeInsightPipeline,
    estimate_processing_time,
    estimate_run_cost,
    calculate_model_cost,
)

print('OOP package loaded successfully.')
print('Default model:', DEFAULT_MODEL)


## 2. Initialise the services

In [ ]:

parser = FeedbackParser()
validator = FeedbackValidator()
context = AssignmentContext()
analyzer = AnalysisEngine()
result_manager = ResultManager()


classifier = LLMClassifier(
    api_key=os.environ.get("TOGETHER_API_KEY"),
    model=DEFAULT_MODEL,
    batch_size=BATCH_SIZE,
)

pipeline = EduCodeInsightPipeline(
    parser=parser,
    validator=validator,
    context=context,
    classifier=classifier,
    analyzer=analyzer,
    result_manager=result_manager,
)




## 3. Validate feedback text in memory (offline example)

Streamlit calls `uploaded.getvalue()` and validates each file. The following *synthetic* example demonstrates the same validator without needing any saved assignment or a Together AI API call.


In [ ]:
sample_feedback = "QUESTION 1: Example\nGrade: 0/1\nRubric Justification:\n☐ Compiles (0/1)\n"
valid, errors = validator.validate(sample_feedback, "ExampleStudent.txt")
print("Sample valid:", valid)
print("Validation issues:", errors)
if valid:
    parsed = parser.parse(sample_feedback, "ExampleStudent.txt")
    print("Parsed student:", parsed.student_id)
    print("Parsed questions:", len(parsed.questions))


## 4. Pre-run time estimate

The live application calculates this from the **actual number of valid uploaded feedback files**. The value below is illustrative only.


In [ ]:
sample_upload_count = 1 if valid else 0
time_estimate = pipeline.estimate_time(sample_upload_count)
print("Illustrative (1 sample) processing time:", time_estimate)



## 5. Pre-run cost estimate

The exact pre-run cost depends on the number of input/output tokens. The GUI can estimate this after deriving a token estimate from the uploaded workload. The helper below exposes the same model-pricing calculation for a supplied token estimate.


In [ ]:
estimated_input_tokens = 0
estimated_output_tokens = 0

if estimated_input_tokens or estimated_output_tokens:
    print(pipeline.estimate_cost(
        estimated_input_tokens,
        estimated_output_tokens,
        model=DEFAULT_MODEL,
    ))
else:
    print("Provide estimated input/output tokens to calculate a pre-run cost.")


## 6. Run the assignment analysis

**Use the Streamlit Analyse page** to upload the three input types, review the prompt/cost estimate, and click **Run Analysis**. Streamlit passes uploaded file text and PDF bytes to the backend; no local input paths are required.


## 7. Analyse and visualise results

After a Streamlit analysis completes, the application saves results and makes them available in its **Dashboard** and **Reports** pages. The notebook does not require uploaded files or previously saved runs.


In [ ]:
print("Open the Dashboard page to inspect uploaded-assignment results.")


## 8. Overall analysis

The **Dashboard** page provides overall analysis across previously processed assignments.


## 9. OOP smoke test

In [ ]:

## Checks that do not make any API requesr.
assert DEFAULT_MODEL == "openai/gpt-oss-120b"
assert MAX_WORKERS == 8
assert BATCH_SIZE == 12
assert set(CATEGORIES) == {
    "Syntax Error",
    "Semantic Error",
    "Best Practice Violation",
    "Rule-Based Error",
}
assert isinstance(pipeline, EduCodeInsightPipeline)
assert isinstance(pipeline.parser, FeedbackParser)
assert isinstance(pipeline.validator, FeedbackValidator)
assert isinstance(pipeline.context, AssignmentContext)
assert isinstance(pipeline.classifier, LLMClassifier)
assert isinstance(pipeline.analyzer, AnalysisEngine)
assert isinstance(pipeline.result_manager, ResultManager)

print("OOP smoke test passed.")
